# P6: The Apprentice Model
## Financial Sentiment Distillation — Demo Notebook

This notebook walks through the complete distillation pipeline step by step,
showing intermediate results and visualizations.

**Setup:** Run this in Google Colab (free GPU) or any environment with PyTorch.

```bash
pip install -r requirements.txt
```

In [ ]:
# ── Environment check ─────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, '..')   # add project root

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')

## 1. Dataset: FinancialPhraseBank

We use the `sentences_allagree` split — 2,264 sentences where **all annotators agreed**
on the sentiment label. This gives us the highest-quality ground truth.

**Labels:** negative (0) · neutral (1) · positive (2)

**Source:** Malo et al. (2014), HuggingFace Hub: `financial_phrasebank`

In [ ]:
from src.data.loader import FinancialPhraseBankLoader

loader = FinancialPhraseBankLoader(seed=42)
split = loader.load()

print('Dataset statistics:')
import json
print(json.dumps(split.stats(), indent=2))

In [ ]:
# Inspect some examples
print('Sample training examples:')
for s in split.train[:5]:
    print(f'  [{s.label_str:8s}] {s.text[:90]}...')

## 2. Teacher Annotation (Mock Mode)

In the full experiment, we call the Claude API to get soft probability labels.
Here we use a **Mock teacher** for demonstration (no API key needed).

The mock teacher adds calibrated noise around the true label, simulating
the confidence distributions a real LLM would produce.

In [ ]:
from src.models.teacher import MockTeacher

teacher = MockTeacher(confidence=0.80, seed=42)
teacher.annotate_batch(split.train, verbose=False)
teacher.annotate_batch(split.val, verbose=False)

# Show soft labels for 5 examples
print('Sample soft labels:')
print(f'{"Label":10s} {"neg":>6} {"neu":>6} {"pos":>6}  Text')
print('-'*80)
for s in split.train[:5]:
    neg, neu, pos = s.soft_labels
    print(f'{s.label_str:10s} {neg:6.3f} {neu:6.3f} {pos:6.3f}  {s.text[:50]}...')

In [ ]:
# Soft label distribution plot
from src.visualization.plots import plot_soft_label_distribution
import matplotlib.pyplot as plt

plot_soft_label_distribution(split.train, output_dir='../outputs/figures')
plt.show()

## 3. Train Baseline Student (Cross-Entropy only)

In [ ]:
from src.models.student import StudentModel, StudentConfig

baseline_config = StudentConfig(
    num_epochs=3,          # reduce for demo (use 5 for full run)
    batch_size=16,
    output_dir='../outputs/student_baseline',
    seed=42,
)

baseline_student = StudentModel(baseline_config)
baseline_history = baseline_student.train(
    split.train, split.val,
    use_distillation=False,
)

## 4. Train Distilled Student (KD + Cross-Entropy)

In [ ]:
distilled_config = StudentConfig(
    num_epochs=3,
    batch_size=16,
    temperature=4.0,   # key distillation hyperparameter
    alpha=0.7,         # 70% KD loss, 30% CE loss
    output_dir='../outputs/student_distilled',
    seed=42,
)

distilled_student = StudentModel(distilled_config)
distilled_history = distilled_student.train(
    split.train, split.val,
    use_distillation=True,
)

## 5. Evaluation & Comparison

In [ ]:
from src.evaluation.metrics import Evaluator

evaluator = Evaluator(output_dir='../outputs/evaluation')

baseline_metrics  = evaluator.evaluate_student(
    baseline_student, split.test,
    model_name='DistilBERT-Baseline', mode='baseline'
)
distilled_metrics = evaluator.evaluate_student(
    distilled_student, split.test,
    model_name='DistilBERT-Distilled', mode='distilled'
)

report = evaluator.compare(baseline_metrics, distilled_metrics)
report.print_summary()

## 6. Visualizations

In [ ]:
from src.visualization.plots import (
    plot_training_curves,
    plot_confusion_matrices,
    plot_model_comparison,
)

fig_dir = '../outputs/figures'

plot_training_curves(baseline_history, distilled_history, output_dir=fig_dir)
plot_confusion_matrices(baseline_metrics.confusion, distilled_metrics.confusion, output_dir=fig_dir)
plot_model_comparison(baseline_metrics, distilled_metrics, output_dir=fig_dir)

plt.show()
print('All figures saved to outputs/figures/')

## 7. Inference Demo

Try the distilled student on your own sentences.

In [ ]:
test_sentences = [
    "The company announced record quarterly profits driven by strong international growth.",
    "Management reiterated its full-year guidance unchanged.",
    "The firm was forced to write down EUR 200 million in goodwill impairments.",
    "Supply chain disruptions continue to weigh on operating margins.",
    "The board approved a EUR 500 million share buyback programme.",
]

results = distilled_student.predict(test_sentences)

print(f'{"Sentence":65s}  {"Label":10s}  {"Conf":6}')
print('-'*90)
for sent, res in zip(test_sentences, results):
    print(f'{sent[:64]:65s}  {res["label"]:10s}  {res["confidence"]:.3f}')